# Machine Learning con Penguins Dataset (Solo Variables Numéricas)
## Clasificación con 4 Features - Enfoque Simplificado

**Enfoque:**
- Solo 4 variables numéricas (igual que Iris)
- Sin codificación de categóricas
- Modelo más simple y directo
- Comparación de rendimiento con el modelo completo (6 features)

In [ ]:
# Importar bibliotecas
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Machine Learning
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn import metrics

# Configuración
%matplotlib inline
import warnings
warnings.filterwarnings('ignore')

## 1. Carga y Preparación de Datos

In [ ]:
# Cargar el dataset
data = sns.load_dataset('penguins')
print(f"Dimensiones originales: {data.shape}")

# Eliminar valores nulos
data_clean = data.dropna()
print(f"Dimensiones después de limpieza: {data_clean.shape}")
print(f"Filas eliminadas: {len(data) - len(data_clean)}")

In [ ]:
# Seleccionar SOLO las variables numéricas + species
columns_to_use = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g', 'species']
data_numeric = data_clean[columns_to_use].copy()

print(f"Columnas seleccionadas: {list(data_numeric.columns)}")
print(f"Dimensiones del dataset: {data_numeric.shape}")
print("\nPrimeras filas:")
data_numeric

## 2. División de Datos: Entrenamiento y Prueba

In [ ]:
# División estratificada (60% entrenamiento, 40% prueba)
train, test = train_test_split(data_numeric, test_size=0.4, 
                                stratify=data_numeric['species'], 
                                random_state=42)

print(f"Total de datos: {len(data_numeric)}")
print(f"Entrenamiento: {len(train)} ({len(train)/len(data_numeric)*100:.1f}%)")
print(f"Prueba: {len(test)} ({len(test)/len(data_numeric)*100:.1f}%)")

print("\nDistribución de especies:")
print("\nEntrenamiento:")
print(train['species'].value_counts())
print("\nPrueba:")
print(test['species'].value_counts())

## 3. Preparación de Features y Target

In [ ]:
# Definir features (variables independientes X)
# SOLO 4 VARIABLES NUMÉRICAS (como en Iris)
feature_names = [
    'bill_length_mm',
    'bill_depth_mm',
    'flipper_length_mm',
    'body_mass_g'
]

# Nombres de las clases (especies)
class_names = ['Adelie', 'Chinstrap', 'Gentoo']

print(f"Total de features: {len(feature_names)}")
print(f"\nLista de features:")
for i, feature in enumerate(feature_names, 1):
    print(f"  {i}. {feature}")

print(f"\nClases a predecir (especies):")
for i, clase in enumerate(class_names, 1):
    print(f"  {i}. {clase}")

In [ ]:
# Preparar conjuntos X (features) e y (target)
X_train = train[feature_names]
y_train = train['species']

X_test = test[feature_names]
y_test = test['species']

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_test shape: {y_test.shape}")

print("\nPrimeras 5 filas de X_train:")
X_train.head()

## 4. Modelo: Árbol de Decisión

In [ ]:
# Crear el modelo (similar al que usaste con Iris)
dt_model = DecisionTreeClassifier(max_depth=4, random_state=42, min_samples_split=5)

# Entrenar el modelo
dt_model.fit(X_train, y_train)

print("✓ Modelo de Árbol de Decisión entrenado exitosamente!")
print(f"\nProfundidad del árbol: {dt_model.get_depth()}")
print(f"Número de hojas: {dt_model.get_n_leaves()}")
print(f"Número de features utilizados: {dt_model.n_features_in_}")

In [ ]:
# Realizar predicciones
y_train_pred = dt_model.predict(X_train)
y_test_pred = dt_model.predict(X_test)

print("✓ Predicciones completadas")
print(f"\nEjemplo de predicciones (primeras 10):")
print(pd.DataFrame({
    'Real': y_test.values[:10],
    'Predicho': y_test_pred[:10]
}))

## 5. Evaluación del Modelo

In [ ]:
# Calcular accuracy
train_accuracy = metrics.accuracy_score(y_train, y_train_pred)
test_accuracy = metrics.accuracy_score(y_test, y_test_pred)

print("=" * 70)
print("EVALUACIÓN DEL MODELO - ÁRBOL DE DECISIÓN (4 FEATURES)")
print("=" * 70)
print(f"\nAccuracy en Entrenamiento: {train_accuracy:.4f} ({train_accuracy*100:.2f}%)")
print(f"Accuracy en Prueba: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")
print(f"\nDiferencia (señal de overfitting): {(train_accuracy - test_accuracy)*100:.2f}%")

if abs(train_accuracy - test_accuracy) < 0.05:
    print("\n✓ Buen balance entre entrenamiento y prueba (< 5% diferencia)")
else:
    print("\n⚠ Posible overfitting (> 5% diferencia)")

In [ ]:
# Matriz de confusión
cm = metrics.confusion_matrix(y_test, y_test_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=class_names, yticklabels=class_names,
            cbar_kws={'label': 'Cantidad'}, linewidths=2, linecolor='white')
plt.title('Matriz de Confusión - Árbol de Decisión (4 Features)', fontsize=14, fontweight='bold', pad=15)
plt.ylabel('Valor Real', fontsize=12)
plt.xlabel('Valor Predicho', fontsize=12)
plt.tight_layout()
plt.show()

print("Matriz de Confusión:")
print(cm)
print(f"\nDiagonal (predicciones correctas): {cm.diagonal().sum()} de {cm.sum()}")

In [ ]:
# Reporte de clasificación detallado
print("=" * 70)
print("REPORTE DE CLASIFICACIÓN DETALLADO")
print("=" * 70)
print(metrics.classification_report(y_test, y_test_pred, target_names=class_names))

# Explicación de métricas
print("\nEXPLICACIÓN DE MÉTRICAS:")
print("-" * 70)
print("Precision: De todas las predicciones de una clase, ¿cuántas fueron correctas?")
print("Recall: De todos los casos reales de una clase, ¿cuántos fueron detectados?")
print("F1-score: Media armónica de precision y recall (balance entre ambos)")
print("Support: Cantidad de casos reales de cada clase en el conjunto de prueba")

## 6. Importancia de Features

In [ ]:
# Obtener importancia de features
feature_importance = dt_model.feature_importances_

# Crear DataFrame para mejor visualización
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': feature_importance
}).sort_values('Importance', ascending=False)

print("=" * 70)
print("IMPORTANCIA DE FEATURES")
print("=" * 70)
print(importance_df.to_string(index=False))
print(f"\nTotal: {importance_df['Importance'].sum():.4f}")

# Identificar feature más importante
top_feature = importance_df.iloc[0]
print(f"\n🏆 Feature más importante: {top_feature['Feature']} ({top_feature['Importance']:.4f})")

In [ ]:
# Visualizar importancia de features
plt.figure(figsize=(12, 6))
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A']
bars = plt.barh(importance_df['Feature'], importance_df['Importance'], color=colors, alpha=0.8, edgecolor='black')
plt.xlabel('Importancia', fontsize=12, fontweight='bold')
plt.ylabel('Feature', fontsize=12, fontweight='bold')
plt.title('Importancia de Features en el Árbol de Decisión', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()

# Añadir valores en las barras
for i, (feature, importance) in enumerate(zip(importance_df['Feature'], importance_df['Importance'])):
    plt.text(importance + 0.01, i, f'{importance:.4f}', va='center', fontweight='bold')

plt.grid(alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

## 7. Visualización del Árbol de Decisión

In [ ]:
# Visualizar el árbol de decisión completo
plt.figure(figsize=(20, 12))
plot_tree(dt_model, 
          feature_names=feature_names,
          class_names=class_names,
          filled=True,
          rounded=True,
          fontsize=10,
          proportion=True)
plt.title('Árbol de Decisión Completo (4 Features Numéricas)', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

In [ ]:
# Árbol simplificado (primeros 2 niveles)
plt.figure(figsize=(16, 10))
plot_tree(dt_model, 
          feature_names=feature_names,
          class_names=class_names,
          filled=True,
          rounded=True,
          fontsize=12,
          max_depth=2)
plt.title('Árbol de Decisión (Primeros 2 Niveles)', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

## 8. Modelo Alternativo: Random Forest

In [ ]:
# Entrenar Random Forest
rf_model = RandomForestClassifier(n_estimators=100, max_depth=4, random_state=42)
rf_model.fit(X_train, y_train)

# Predicciones
y_train_pred_rf = rf_model.predict(X_train)
y_test_pred_rf = rf_model.predict(X_test)

# Evaluación
rf_train_accuracy = metrics.accuracy_score(y_train, y_train_pred_rf)
rf_test_accuracy = metrics.accuracy_score(y_test, y_test_pred_rf)

print("=" * 70)
print("COMPARACIÓN DE MODELOS (4 FEATURES NUMÉRICAS)")
print("=" * 70)
print(f"\nÁrbol de Decisión:")
print(f"  Entrenamiento: {train_accuracy:.4f} ({train_accuracy*100:.2f}%)")
print(f"  Prueba: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")

print(f"\nRandom Forest:")
print(f"  Entrenamiento: {rf_train_accuracy:.4f} ({rf_train_accuracy*100:.2f}%)")
print(f"  Prueba: {rf_test_accuracy:.4f} ({rf_test_accuracy*100:.2f}%)")

print(f"\nMejora con Random Forest: {(rf_test_accuracy - test_accuracy)*100:.2f}%")

In [ ]:
# Comparar importancia de features entre modelos
rf_importance = rf_model.feature_importances_

comparison_df = pd.DataFrame({
    'Feature': feature_names,
    'DT_Importance': feature_importance,
    'RF_Importance': rf_importance,
    'Diferencia': rf_importance - feature_importance
}).sort_values('RF_Importance', ascending=False)

print("\nCOMPARACIÓN DE IMPORTANCIA DE FEATURES:")
print("=" * 70)
print(comparison_df.to_string(index=False))

In [ ]:
# Visualización comparativa
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Decision Tree
axes[0].barh(comparison_df['Feature'], comparison_df['DT_Importance'], 
             color='#4ECDC4', alpha=0.8, edgecolor='black')
axes[0].set_xlabel('Importancia', fontsize=11, fontweight='bold')
axes[0].set_title('Árbol de Decisión', fontsize=13, fontweight='bold')
axes[0].invert_yaxis()
axes[0].grid(alpha=0.3, axis='x')

# Random Forest
axes[1].barh(comparison_df['Feature'], comparison_df['RF_Importance'], 
             color='#FF6B6B', alpha=0.8, edgecolor='black')
axes[1].set_xlabel('Importancia', fontsize=11, fontweight='bold')
axes[1].set_title('Random Forest', fontsize=13, fontweight='bold')
axes[1].invert_yaxis()
axes[1].grid(alpha=0.3, axis='x')

plt.suptitle('Comparación de Importancia de Features', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 9. Validación Cruzada

In [ ]:
# Validación cruzada con 5 folds
cv_scores_dt = cross_val_score(dt_model, X_train, y_train, cv=5, scoring='accuracy')
cv_scores_rf = cross_val_score(rf_model, X_train, y_train, cv=5, scoring='accuracy')

print("=" * 70)
print("VALIDACIÓN CRUZADA (5-Fold)")
print("=" * 70)

print(f"\nÁrbol de Decisión:")
print(f"  Scores por fold: {cv_scores_dt}")
print(f"  Promedio: {cv_scores_dt.mean():.4f}")
print(f"  Desviación estándar: {cv_scores_dt.std():.4f}")
print(f"  Rango: [{cv_scores_dt.min():.4f}, {cv_scores_dt.max():.4f}]")

print(f"\nRandom Forest:")
print(f"  Scores por fold: {cv_scores_rf}")
print(f"  Promedio: {cv_scores_rf.mean():.4f}")
print(f"  Desviación estándar: {cv_scores_rf.std():.4f}")
print(f"  Rango: [{cv_scores_rf.min():.4f}, {cv_scores_rf.max():.4f}]")

In [ ]:
# Visualizar scores de validación cruzada
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Gráfico de barras
x_pos = np.arange(5)
width = 0.35

axes[0].bar(x_pos - width/2, cv_scores_dt, width, label='Decision Tree', alpha=0.8, color='#4ECDC4', edgecolor='black')
axes[0].bar(x_pos + width/2, cv_scores_rf, width, label='Random Forest', alpha=0.8, color='#FF6B6B', edgecolor='black')
axes[0].set_xlabel('Fold', fontsize=12)
axes[0].set_ylabel('Accuracy', fontsize=12)
axes[0].set_title('Validación Cruzada por Fold', fontsize=13, fontweight='bold')
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels([f'Fold {i+1}' for i in range(5)])
axes[0].legend()
axes[0].set_ylim([0.9, 1.0])
axes[0].grid(alpha=0.3, axis='y')

# Box plot
axes[1].boxplot([cv_scores_dt, cv_scores_rf], labels=['Decision Tree', 'Random Forest'],
                patch_artist=True, boxprops=dict(facecolor='lightblue', alpha=0.7))
axes[1].set_ylabel('Accuracy', fontsize=12)
axes[1].set_title('Distribución de Scores CV', fontsize=13, fontweight='bold')
axes[1].set_ylim([0.9, 1.0])
axes[1].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 10. Análisis de Predicciones

In [ ]:
# Crear DataFrame con resultados
results = test.copy()
results['predicted'] = y_test_pred
results['correct'] = results['species'] == results['predicted']

print(f"Total de predicciones: {len(results)}")
print(f"Predicciones correctas: {results['correct'].sum()} ({results['correct'].mean()*100:.2f}%)")
print(f"Predicciones incorrectas: {(~results['correct']).sum()} ({(~results['correct']).mean()*100:.2f}%)")

In [ ]:
# Analizar casos mal clasificados
misclassified = results[~results['correct']]

if len(misclassified) > 0:
    print(f"\n{'='*70}")
    print(f"CASOS MAL CLASIFICADOS ({len(misclassified)})")
    print(f"{'='*70}")
    print(misclassified[['species', 'predicted'] + feature_names].to_string())
    
    # Análisis de errores por especie
    print(f"\n{'='*70}")
    print("DISTRIBUCIÓN DE ERRORES POR ESPECIE:")
    print(f"{'='*70}")
    error_counts = misclassified.groupby(['species', 'predicted']).size()
    print(error_counts)
else:
    print("\n🎉 ¡PERFECTO! Todas las predicciones fueron correctas.")

In [ ]:
# Visualizar predicciones correctas vs incorrectas
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Gráfico 1: Bill dimensions
correct_color = results['correct'].map({True: 'green', False: 'red'})
scatter1 = axes[0].scatter(results['bill_length_mm'], results['bill_depth_mm'], 
                           c=correct_color, s=100, alpha=0.6, edgecolors='black', linewidth=1.5)
axes[0].set_xlabel('Bill Length (mm)', fontsize=11)
axes[0].set_ylabel('Bill Depth (mm)', fontsize=11)
axes[0].set_title('Predicciones: Bill Dimensions', fontsize=13, fontweight='bold')
axes[0].grid(alpha=0.3)

# Leyenda manual
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='green', label='Correctas'),
    Patch(facecolor='red', label='Incorrectas')
]
axes[0].legend(handles=legend_elements, loc='best')

# Gráfico 2: Body dimensions
scatter2 = axes[1].scatter(results['flipper_length_mm'], results['body_mass_g'], 
                           c=correct_color, s=100, alpha=0.6, edgecolors='black', linewidth=1.5)
axes[1].set_xlabel('Flipper Length (mm)', fontsize=11)
axes[1].set_ylabel('Body Mass (g)', fontsize=11)
axes[1].set_title('Predicciones: Body Dimensions', fontsize=13, fontweight='bold')
axes[1].grid(alpha=0.3)
axes[1].legend(handles=legend_elements, loc='best')

plt.tight_layout()
plt.show()

## 11. Predicción de Nuevos Datos

In [ ]:
# Ejemplo: Predecir especie de un nuevo pingüino
nuevo_pinguino = pd.DataFrame({
    'bill_length_mm': [45.0],
    'bill_depth_mm': [15.5],
    'flipper_length_mm': [195],
    'body_mass_g': [3800]
})

print("DATOS DEL NUEVO PINGÜINO:")
print("=" * 70)
print(nuevo_pinguino)

# Predicción
prediccion = dt_model.predict(nuevo_pinguino)
probabilidades = dt_model.predict_proba(nuevo_pinguino)

print(f"\n{'='*70}")
print(f"PREDICCIÓN: {prediccion[0]}")
print(f"{'='*70}")

print(f"\nPROBABILIDADES POR ESPECIE:")
for especie, prob in zip(class_names, probabilidades[0]):
    bar = '█' * int(prob * 50)
    print(f"  {especie:12s}: {prob:.4f} ({prob*100:5.2f}%) {bar}")

# Interpretación
max_prob = probabilidades[0].max()
if max_prob > 0.9:
    print(f"\n✓ Alta confianza en la predicción ({max_prob*100:.1f}%)")
elif max_prob > 0.7:
    print(f"\n⚠ Confianza moderada en la predicción ({max_prob*100:.1f}%)")
else:
    print(f"\n⚠ Baja confianza en la predicción ({max_prob*100:.1f}%)")

In [ ]:
# Crear función reutilizable para predicciones
def predecir_especie(bill_length, bill_depth, flipper_length, body_mass):
    """
    Predice la especie de un pingüino dados sus parámetros físicos.
    
    Parámetros:
    - bill_length: Longitud del pico en mm
    - bill_depth: Profundidad del pico en mm
    - flipper_length: Longitud de la aleta en mm
    - body_mass: Masa corporal en gramos
    
    Retorna:
    - Especie predicha y probabilidades
    """
    datos = pd.DataFrame({
        'bill_length_mm': [bill_length],
        'bill_depth_mm': [bill_depth],
        'flipper_length_mm': [flipper_length],
        'body_mass_g': [body_mass]
    })
    
    prediccion = dt_model.predict(datos)[0]
    probabilidades = dt_model.predict_proba(datos)[0]
    
    print(f"Predicción: {prediccion}")
    print(f"Probabilidades:")
    for especie, prob in zip(class_names, probabilidades):
        print(f"  {especie}: {prob:.4f} ({prob*100:.2f}%)")
    
    return prediccion, probabilidades

# Ejemplo de uso
print("EJEMPLO 1: Pingüino pequeño")
print("=" * 70)
predecir_especie(38.0, 18.5, 185, 3200)

print(f"\n{'='*70}")
print("EJEMPLO 2: Pingüino grande")
print("=" * 70)
predecir_especie(50.0, 15.0, 220, 5500)

## 12. Resumen Final

In [ ]:
print("=" * 80)
print("RESUMEN FINAL DEL MODELO (4 FEATURES NUMÉRICAS)")
print("=" * 80)

print(f"\n1. CONFIGURACIÓN DEL MODELO:")
print(f"   - Algoritmo: Árbol de Decisión")
print(f"   - Profundidad máxima: {dt_model.max_depth}")
print(f"   - Profundidad real: {dt_model.get_depth()}")
print(f"   - Número de hojas: {dt_model.get_n_leaves()}")

print(f"\n2. DATOS:")
print(f"   - Total de registros (limpios): {len(data_numeric)}")
print(f"   - Entrenamiento: {len(train)} ({len(train)/len(data_numeric)*100:.1f}%)")
print(f"   - Prueba: {len(test)} ({len(test)/len(data_numeric)*100:.1f}%)")

print(f"\n3. FEATURES UTILIZADAS (4 NUMÉRICAS):")
for i, feature in enumerate(feature_names, 1):
    importance = importance_df[importance_df['Feature'] == feature]['Importance'].values[0]
    print(f"   {i}. {feature:20s} - Importancia: {importance:.4f}")

print(f"\n4. RENDIMIENTO DEL MODELO:")
print(f"   Árbol de Decisión:")
print(f"     * Accuracy en entrenamiento: {train_accuracy:.4f} ({train_accuracy*100:.2f}%)")
print(f"     * Accuracy en prueba: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")
print(f"     * CV Score promedio: {cv_scores_dt.mean():.4f} ± {cv_scores_dt.std():.4f}")
print(f"\n   Random Forest:")
print(f"     * Accuracy en prueba: {rf_test_accuracy:.4f} ({rf_test_accuracy*100:.2f}%)")
print(f"     * CV Score promedio: {cv_scores_rf.mean():.4f} ± {cv_scores_rf.std():.4f}")

print(f"\n5. FEATURE MÁS IMPORTANTE:")
top_feature = importance_df.iloc[0]
print(f"   🏆 {top_feature['Feature']}: {top_feature['Importance']:.4f} ({top_feature['Importance']*100:.2f}%)")

print(f"\n6. ERRORES DE CLASIFICACIÓN:")
print(f"   - Total de errores: {(~results['correct']).sum()} de {len(results)}")
print(f"   - Tasa de error: {(~results['correct']).mean()*100:.2f}%")

print(f"\n7. CONCLUSIONES:")
if test_accuracy >= 0.95:
    print(f"   ✓ Excelente rendimiento (>95% accuracy)")
elif test_accuracy >= 0.90:
    print(f"   ✓ Muy buen rendimiento (>90% accuracy)")
else:
    print(f"   ⚠ Rendimiento aceptable pero mejorable")

if abs(train_accuracy - test_accuracy) < 0.05:
    print(f"   ✓ Sin overfitting significativo")
else:
    print(f"   ⚠ Posible overfitting detectado")

print(f"   ✓ Las 4 variables numéricas son suficientes para clasificar especies")
print(f"   ✓ Modelo simple, interpretable y eficiente")

print("\n" + "=" * 80)

In [ ]:
# Guardar el modelo entrenado
import pickle

with open('penguin_model_4features.pkl', 'wb') as f:
    pickle.dump(dt_model, f)

with open('penguin_rf_model_4features.pkl', 'wb') as f:
    pickle.dump(rf_model, f)

print("✓ Modelos guardados exitosamente!")
print("  - penguin_model_4features.pkl (Árbol de Decisión)")
print("  - penguin_rf_model_4features.pkl (Random Forest)")
print("\nPuedes cargarlos posteriormente con:")
print("  with open('penguin_model_4features.pkl', 'rb') as f:")
print("      modelo = pickle.load(f)")